## Aeropulse — Gold: Origin Airport Dimension

**Purpose:** Builds `dim_origin_airport` from silver `airport`, enriched with `origin_city_name` looked up from silver `flight` (airport reference data alone doesn't carry the city name used on the flight record). Writes the result as `origin_airport_sk` keyed rows.

**Batch parameters:** `batch_id`, `batch_year` — used to filter the silver `flight` lookup to the current batch

**Depends on:** `gold-environment`, `gold-helper` (run via `%run`)

**Reads:** `silver.airport`, `silver.flight` (both filtered to `batch_id` where relevant)

**Writes:** `dim_origin_airport` (merge on `origin_airport_sk`)


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

StatementMeta(, 258ed49b-ed69-49e4-85cc-910ee2a9dd66, 3, Finished, Available, Finished, False)

In [2]:
batch_id = ""
batch_year = ""

StatementMeta(, 258ed49b-ed69-49e4-85cc-910ee2a9dd66, 4, Finished, Available, Finished, False)

In [3]:
%run gold-environment

StatementMeta(, 258ed49b-ed69-49e4-85cc-910ee2a9dd66, 5, Finished, Available, Finished, True)

In [4]:
%run gold-helper

StatementMeta(, 258ed49b-ed69-49e4-85cc-910ee2a9dd66, 8, Finished, Available, Finished, True)

## Read airport df

In [5]:
airport_df = spark.read.format('delta').load(airport_silver_path).filter(F.col('batch_id')==batch_id)

StatementMeta(, 258ed49b-ed69-49e4-85cc-910ee2a9dd66, 9, Finished, Available, Finished, False)

## dropping unecessary columns for dim_airport

In [6]:
airport_df = airport_df.drop('ingested_timestamp', 'source_path')

##display(airport_df.limit(10))

StatementMeta(, 258ed49b-ed69-49e4-85cc-910ee2a9dd66, 10, Finished, Available, Finished, False)

## read flight silver

In [7]:
flight_df = spark.read.format('delta').load(flight_silver_path).filter(F.col("batch_id")==batch_id)

StatementMeta(, 258ed49b-ed69-49e4-85cc-910ee2a9dd66, 11, Finished, Available, Finished, False)

## retrieving information about airport from flight silver dataframe
 - drop this retrieved attribute from flight df in gold layer
 - retrieve:
   - origin_city_name
   - 

In [8]:
## deduplicating on flight_df
flight_lookup = flight_df.select(
    F.col("origin_airport_code"),
    F.col("origin_city_name")
).dropDuplicates(["origin_airport_code"])


## join deplicate table to airport df

airport_join = airport_df.alias('a').join(
    flight_lookup.alias('f'), F.col('a.airport_code') == F.col('f.origin_airport_code'), 
    how="left"
    ).select(
        F.col('a.airport_code').alias("origin_airport_code"),
        F.col('a.airport_description'),
        F.col('a.airport_name'),
        F.col('a.airport_city'),
        F.col('a.airport_state'),
        F.col('a.airport_sk').alias("origin_airport_sk"),
        F.col('a.created_timestamp'),
        F.col('a.updated_timestamp'),
        F.col('f.origin_city_name').alias('airport_city_name')
)

StatementMeta(, 258ed49b-ed69-49e4-85cc-910ee2a9dd66, 12, Finished, Available, Finished, False)

before join: 6818 | after join: 6818


## write to gold layer as dim_origin_airport

In [9]:
# write to gold layer

update_cols = [c for c in airport_join.columns if c not in ["origin_airport_sk"]]

write_to_gold(
    airport_join,
    "dim_origin_airport",
    "s.origin_airport_sk= t.origin_airport_sk",
    update_cols
)


StatementMeta(, 258ed49b-ed69-49e4-85cc-910ee2a9dd66, 13, Finished, Available, Finished, False)